# Replay

In [ ]:
from typing import TypedDict

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from loguru import logger

load_dotenv(override=True)
from rich import print
import os

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)


#1. 状態を宣言
class OverAllState(TypedDict):
    topic: str
    poem: str
    joke: str
    final_output: str


#1.1 入力状態
class InputState(TypedDict):
    topic: str


#1.2 出力状態
class OutputState(TypedDict):
    final_output: str


topics = ["ラグドール", "三毛猫", "キジトラ"]
topic_index = 0


#2. ノードを定義
def node_change_topic(state: InputState) -> OverAllState:
    global topic_index
    logger.info("topic_index:{}", topic_index)
    sub_topic = topics[topic_index]
    topic_index += 1
    topic_index %= len(topics)

    return {
        "topic": f"{state["topic"]}:{sub_topic}"
    }


#2.1 同じスーパーステップ内で正常に実行されるノード
def node_poem(state: OverAllState) -> OverAllState:
    logger.info("node_poem を実行中")
    topic = state["topic"]
    poem = model.invoke([HumanMessage(f"{topic}をテーマにした俳句を詠んでください")]).content
    return {
        "poem": poem
    }


import time


#2.2 同じスーパーステップ内で実行に失敗するノード
def node_joke(state: OverAllState) -> OverAllState:
    logger.info("node_joke を実行中")
    topic = state["topic"]
    # time.sleep(5)
    # raise Exception("意図的に例外をスロー")
    joke = model.invoke([HumanMessage(f"{topic}をテーマにしたジョークを書いてください")]).content
    return {
        "joke": joke
    }


def node_output(state: OverAllState) -> OutputState:
    logger.info("node_output を実行中")
    topic = state["topic"]
    poem = state["poem"]
    joke = state["joke"]
    final_output = f"{topic}に関する俳句:{poem}\n ジョーク:{joke}\n"
    return {
        "final_output": final_output
    }


#3. グラフを構築
builder = StateGraph(state_schema=OverAllState, input_schema=InputState, output_schema=OutputState)

#3.1 ノードを追加
builder.add_node("node_change_topic", node_change_topic)
builder.add_node("node_poem", node_poem)
builder.add_node("node_joke", node_joke)
builder.add_node("node_output", node_output)

#3.2 エッジを追加
builder.add_edge(START, "node_change_topic")
builder.add_edge("node_change_topic", "node_poem")
builder.add_edge("node_change_topic", "node_joke")
builder.add_edge("node_poem", "node_output")
builder.add_edge("node_joke", "node_output")
builder.add_edge("node_output", END)

#4. チェックポイントバックエンドを追加
DB_URL = os.getenv("DB_URL")
from langgraph.checkpoint.postgres import PostgresSaver

with PostgresSaver.from_conn_string(DB_URL) as checkpointer:
    #5. PostgresSaver を初めてチェックポイントとして使用する場合は setup() メソッドを呼び出す必要がある
    #checkpointer.setup()
    graph = builder.compile(checkpointer=checkpointer)

    from IPython.display import display

    display(graph)

    config = {
        "configurable": {
            "thread_id": "chapter03-08"
        }
    }

    res = graph.invoke({"topic": "猫"}, config=config)
    print(res)


In [ ]:
with PostgresSaver.from_conn_string(DB_URL) as checkpointer:
    #5. PostgresSaver を初めてチェックポイントとして使用する場合は setup() メソッドを呼び出す必要がある
    #checkpointer.setup()
    graph = builder.compile(checkpointer=checkpointer)

    from IPython.display import display

    display(graph)

    config = {
        "configurable": {
            "thread_id": "chapter03-08"
        }
    }

    res = graph.invoke({"topic": "猫"}, config=config)
    print(res)

In [ ]:
with PostgresSaver.from_conn_string(DB_URL) as checkpointer:
    #5. PostgresSaver を初めてチェックポイントとして使用する場合は setup() メソッドを呼び出す必要がある
    #checkpointer.setup()
    graph = builder.compile(checkpointer=checkpointer)

    config = {
        "configurable": {
            "thread_id": "chapter03-08"
        }
    }
    # チェックポイント履歴を取得
    history_checkpoints = list(graph.get_state_history(config=config))

    new_checkpoint = None
    #next = ('node_poem', 'node_joke')
    for checkpoint in history_checkpoints:
        if checkpoint.next == ('node_poem', 'node_joke'):
            new_checkpoint = checkpoint
            break

    # replay の効果を実現したい場合は、状態に None を指定し、config には以前のいずれかのチェックポイントの config を指定する
    res = graph.invoke(None, config=new_checkpoint.config)
    print(res)